# Quickstart

Run the arousal dynamics model on a week of office light, read its outputs, and vary a parameter. CI executes this notebook, so it stays in step with the package.

In [ ]:
from dataclasses import replace

import numpy as np

from arousal_dynamics import (
    ArousalModel,
    ArousalParameters,
    Inputs,
    LightInput,
    LightUnit,
    WakeSchedule,
    simulate,
)
from arousal_dynamics.plotting import plot_light, plot_trajectory

HOUR = 3600.0
DAY = 24 * HOUR

## Light and schedule

Light is a series with a unit. Here: 5 lux of melanopic EDI at night, 250 lux at the office from 08:00 to 18:00, and 30 lux in the evening. `previous` interpolation holds each level until the next sample, which suits a schedule; `cubic` reproduces the original `model_run`. The forced-wake window keeps the model awake from 07:00 to 23:00.

In [ ]:
days = 7
hours = np.arange(0.0, (days + 1) * 24.0, 0.25)
clock = hours % 24
lux = np.select(
    [(clock >= 8) & (clock < 18), (clock >= 7) & (clock < 23)], [250.0, 30.0], default=5.0
)
light = LightInput.from_samples(
    hours * HOUR, lux, unit=LightUnit.MELANOPIC_EDI, interpolation="previous"
)
inputs = Inputs(light, WakeSchedule(waketime=7, bedtime=23))

## Simulate

`simulate` takes the model, its inputs, and the output times in seconds. The result holds every state and the derived outputs (KSS, sleep state, circadian phase, aMT6s), each by name.

In [ ]:
run = simulate(ArousalModel(), inputs, np.arange(0.0, days * DAY + 1, 600.0))
print(run.names)
awake = run["S"] == 1
print(f"mean KSS while awake: {run['KSS'][awake].mean():.2f}")

In [ ]:
plot_light(run);

In [ ]:
plot_trajectory(run, ["KSS", "H", "Theta_L", "rho_b", "S"]);

## Vary a parameter

Parameters are a frozen dataclass with the papers' values as defaults; override any of them per run. This one moves the midpoint of the light-dependent alertness sigmoid (Tekieh 2020, Eq 14), which `examples/sigmoid.py` used to need a copy of the model for.

In [ ]:
refit = replace(ArousalParameters(), s_b=0.3)
variant = simulate(ArousalModel(), inputs, run.t, params=refit)
print(f"mean KSS while awake, s_b = 0.3: {variant['KSS'][awake].mean():.2f}")

## As a table

`to_dataframe` needs pandas (`arousal-dynamics[pandas]`).

In [ ]:
run.to_dataframe().iloc[::36, :].head()